In [ ]:
import os
import psycopg2
from dotenv import load_dotenv
from pgvector.psycopg2 import register_vector
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from openai import OpenAI
from langchain_openai import ChatOpenAI

load_dotenv()

In [ ]:
client = OpenAI()

In [ ]:
conn = psycopg2.connect(
    host=os.getenv("POSTGRES_HOST"),
    database=os.getenv("POSTGRES_DB"),
    user=os.getenv("POSTGRES_USER"),
    password=os.getenv("POSTGRES_PASSWORD")
)

In [29]:
register_vector(conn)
cur = conn.cursor()

In [31]:
# Load resume
loader = PyPDFLoader("resume.pdf")
docs = loader.load()

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(docs)

[Document(metadata={'producer': 'pdfTeX-1.40.27', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-01-26T19:53:17+00:00', 'author': '', 'keywords': '', 'moddate': '2026-01-26T19:53:17+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.27 (TeX Live 2025) kpathsea version 6.4.1', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'resume.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}, page_content='Sampada Pokharel\n617 – 230 – 8188|pokharelsampada@gmail.com|linkedin.com/in/pokharelsampada\nExperience\nGalatea Associates St. Petersburg, FL\nConsultant, Embedded Payments: Product & Client Solutions Jul. 2025 – Present\n• Act as technical advisor for embedded payments integrations, supporting onboarding, servicing, and notification workflows for\na B2B SaaS platform supporting 1,000+ SMBs'),
 Document(metadata={'producer': 'pdfTeX-1.40.27', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-01-26T19:53:17+00:00', 'author': '', 'keywords': ''

In [58]:
conn.autocommit = True

In [59]:
for i, chunk in enumerate(chunks, 1):
    text = chunk.page_content
    try:
        emb = client.embeddings.create(
            model="text-embedding-3-small",
            input=text
        ).data[0].embedding
        
        cur.execute(
            "INSERT INTO resume_embeddings (chunk, embedding) VALUES (%s, %s)",
            (text, emb)
        )
        print(f"Inserted chunk {i}/{len(chunks)}")
    except Exception as e:
        print(f"Skipped chunk {i} due to error: {e}")
        conn.rollback()  # clears any failed transaction

Inserted chunk 1/10
Inserted chunk 2/10
Inserted chunk 3/10
Inserted chunk 4/10
Inserted chunk 5/10
Inserted chunk 6/10
Inserted chunk 7/10
Inserted chunk 8/10
Inserted chunk 9/10
Inserted chunk 10/10
